# AGV Tracker — Founding rate analysis

Companion analysis notebook for the AGV Tracker dataset (CLAUDE.md §9 Task 13). Loads the canonical `data/*.csv`, reproduces the dashboard's headline charts, and fits a classic density-dependent Poisson founding-rate model following Hannan & Freeman (1977, 1989) against the three §6.2 sensitivity views.

**Runs end-to-end from CSV alone** (Task 13 Done-when #1). Includes explicit inferential tests — a likelihood-ratio test comparing the full density-dependent GLM to an intercept-only null model, plus the per-coefficient Wald z-tests that `statsmodels.GLM.summary()` emits (Task 13 Done-when #2).

> Small-sample caveats apply throughout (n≈12 years × 3 views). Read coefficient magnitudes as illustrative; cross-validation and a longer time series belong to the SSRN companion paper.

In [ ]:
# Imports and locate the repo root from wherever the notebook is launched.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while not (REPO_ROOT / 'data' / 'agv.csv').exists():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise FileNotFoundError(
            'data/agv.csv not found walking up from ' + str(cwd)
        )
    REPO_ROOT = parent
DATA = REPO_ROOT / 'data'
FIGURES = REPO_ROOT / 'notebooks' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (8.5, 4.5),
    'figure.dpi': 110,
    'font.family': 'serif',
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.frameon': False,
})
pd.options.display.max_columns = 40
print(f'REPO_ROOT = {REPO_ROOT}')


## 1. Data loading

The v0.3 dataset is five relational CSVs under `data/` (CLAUDE.md §4.1). For founding-rate analysis we need the master table plus the lifecycle event log (to distinguish `terminated` AGVs, which are excluded from the "alive at year Y" counts).

In [ ]:
agv = pd.read_csv(DATA / 'agv.csv')
lifecycle = pd.read_csv(DATA / 'agv_lifecycle.csv')

# Extract calendar year from the ISO founded_date.
agv['founded_year'] = (
    pd.to_datetime(agv['founded_date'], errors='coerce').dt.year
)

print(f"{len(agv)} AGVs · {agv['entity_type'].nunique()} entity_types · "
      f"{agv['founded_year'].min()}–{agv['founded_year'].max()}")

agv[['agv_id', 'name_en', 'entity_type',
     'convening_frequency', 'founded_year', 'current_state']].head(5)


## 2. Three sensitivity views (§6.2)

Following the dashboard's toggle, we split the population three ways:

- **continuous**: only venues expected to exist between meetings — `entity_type` in the eight "institution-building" categories.
- **recurring**: only venues whose `convening_frequency` is `annual` or `biennial` (conferences + summit series).
- **one_off_included**: the full dataset, one-off summits included.


In [ ]:
CONTINUOUS_TYPES = {
    'igo_initiative', 'treaty_body', 'multistakeholder_coalition',
    'industry_consortium', 'intl_ngo_thinktank', 'academic_consortium',
    'standards_body_wg', 'national_regulator_intl',
}
RECURRING_FREQUENCIES = {'annual', 'biennial'}

VIEWS = {
    'continuous':       agv[agv['entity_type'].isin(CONTINUOUS_TYPES)].copy(),
    'recurring':        agv[agv['convening_frequency'].isin(RECURRING_FREQUENCIES)].copy(),
    'one_off_included': agv.copy(),
}

pd.Series({name: len(df) for name, df in VIEWS.items()},
          name='n_venues')


## 3. Headline charts

Reproduces the dashboard's Section 1 (cumulative active population) and Section 2 (annual founding rate) from CSV alone, so a reader without browser access can reach the same visual conclusions.

In [ ]:
YEAR_MIN = int(np.nanmin(agv['founded_year']))
YEAR_MAX = max(int(np.nanmax(agv['founded_year'])),
               pd.Timestamp.utcnow().year)
YEARS = np.arange(YEAR_MIN, YEAR_MAX + 1)
print(f'YEARS span: {YEAR_MIN}..{YEAR_MAX} ({len(YEARS)} years)')


In [ ]:
def cumulative_alive(df: pd.DataFrame, years: np.ndarray) -> np.ndarray:
    """Number of AGVs 'alive at year y' under the Task-9 convention:
    founded on or before y AND not yet in current_state='terminated'.
    `absorbed` and `succeeded` count as alive — their work continues
    in the successor record.
    """
    is_active = df['current_state'] != 'terminated'
    alive = np.empty(len(years), dtype=int)
    for i, y in enumerate(years):
        alive[i] = int(((df['founded_year'] <= y) & is_active).sum())
    return alive

fig, ax = plt.subplots()
for name, df in VIEWS.items():
    ax.plot(YEARS, cumulative_alive(df, YEARS), label=name, linewidth=2)
ax.set_title('Cumulative active AGV population (alive at year Y)')
ax.set_xlabel('Year')
ax.set_ylabel('Venues alive')
ax.legend(title='view')
plt.tight_layout()
plt.savefig(FIGURES / 'cumulative_active_population.pdf',
            bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3),
                          sharex=True, sharey=True)
for ax, (name, df) in zip(axes, VIEWS.items()):
    counts = (df.groupby('founded_year').size()
                .reindex(YEARS, fill_value=0))
    ax.bar(counts.index, counts.values, color='steelblue', alpha=0.8,
           edgecolor='white', linewidth=0.6)
    ax.set_title(name.replace('_', ' '))
    ax.set_xlabel('Year')
axes[0].set_ylabel('New AGV foundings')
fig.suptitle('Annual founding rate across the three §6.2 views',
             y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'founding_rate_by_view.pdf', bbox_inches='tight')
plt.show()


## 4. Density-dependent founding rate (Hannan & Freeman)

Population ecology predicts that the founding rate of a population depends on its density through two competing mechanisms:

- **Legitimation** (early phase): as density grows, the venue form becomes "taken for granted", reducing the cost of establishing another one → founding rate rises.
- **Competition** (late phase): once the niche fills, each additional incumbent reduces available resources for new entrants → founding rate falls.

The canonical log-linear Poisson model is

$$\log \lambda_t = \alpha + \beta_1 N_{t-1} + \beta_2 N_{t-1}^2$$

with $\lambda_t$ the expected number of foundings in year $t$ and $N_{t-1}$ the prior-year population density. The textbook prediction is $\beta_1 > 0$ and $\beta_2 < 0$.

In [ ]:
def density_dependence_panel(df: pd.DataFrame,
                              years: np.ndarray) -> pd.DataFrame:
    """Return a (year, density_prev, foundings) DataFrame."""
    density_prev = cumulative_alive(df, years - 1)
    foundings = (
        df.groupby('founded_year').size()
          .reindex(years, fill_value=0).values
    )
    out = pd.DataFrame({
        'year': years,
        'density_prev': density_prev,
        'density_prev_sq': density_prev ** 2,
        'foundings': foundings,
    })
    # Drop the first year so density_prev is well-defined and > 0.
    return out[out['density_prev'] > 0].reset_index(drop=True)

dd_full = density_dependence_panel(VIEWS['one_off_included'], YEARS)
dd_full


In [ ]:
model_full = smf.glm(
    'foundings ~ density_prev + density_prev_sq',
    data=dd_full,
    family=sm.families.Poisson(),
).fit()
print(model_full.summary())


The first-order term and its squared term together capture the classic non-monotone density-dependence shape. With only ~12 years of observations, individual coefficient p-values from the Wald z-test should be read with caution; the likelihood-ratio test in §6 is the more informative inferential check.

## 5. Three-view sensitivity analysis

Refit the same density-dependence model under each of the three views. Coefficient stability across views is the evidence we have that the pattern is not an artefact of a particular definition of 'the population'.

In [ ]:
fits = {}
for name, df in VIEWS.items():
    panel = density_dependence_panel(df, YEARS)
    fits[name] = smf.glm(
        'foundings ~ density_prev + density_prev_sq',
        data=panel,
        family=sm.families.Poisson(),
    ).fit()

coef_table = (
    pd.DataFrame({name: fit.params for name, fit in fits.items()})
      .T.round(4)
)
coef_table.columns = ['α (intercept)', 'β₁ (density_prev)',
                      'β₂ (density_prev²)']
coef_table


In [ ]:
se_table = (
    pd.DataFrame({name: fit.bse for name, fit in fits.items()})
      .T.round(4)
)
se_table.columns = ['SE α', 'SE β₁', 'SE β₂']
se_table


## 6. Inferential test — likelihood-ratio vs. null model

Explicit inferential test: does the density-dependent model improve on an intercept-only Poisson null? The likelihood-ratio statistic is

$$\text{LR} = 2\left(\ell_{\text{full}} - \ell_{\text{null}}\right) \;\sim\; \chi^2_2$$

under the null that $\beta_1 = \beta_2 = 0$.

In [ ]:
null_model = smf.glm(
    'foundings ~ 1',
    data=dd_full,
    family=sm.families.Poisson(),
).fit()

lr_stat = 2.0 * (model_full.llf - null_model.llf)
df_diff = int(model_full.df_model - null_model.df_model)
p_value = float(chi2.sf(lr_stat, df_diff))

print('Likelihood-ratio test — density-dependent vs intercept-only')
print(f'  LR statistic : {lr_stat:.3f}')
print(f'  df           : {df_diff}')
print(f'  p-value      : {p_value:.4f}')
print()
print(f'  ℓ_full = {model_full.llf:.3f}   '
      f'ℓ_null = {null_model.llf:.3f}')


## 7. Publication-ready summary figure

One figure, three panels: observed annual foundings (bars) plus the fitted density-dependent Poisson mean (line) for each view. The PDF is saved to `notebooks/figures/density_dependence_poisson.pdf`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3),
                          sharex=True, sharey=True)
for ax, (name, fit) in zip(axes, fits.items()):
    panel = density_dependence_panel(VIEWS[name], YEARS)
    ax.bar(panel['year'], panel['foundings'], color='#cbd5e1',
           edgecolor='white', linewidth=0.6, label='observed')
    ax.plot(panel['year'], fit.predict(panel),
            color='#0f766e', linewidth=2, label='Poisson fit')
    ax.set_title(name.replace('_', ' '))
    ax.set_xlabel('Year')
axes[0].set_ylabel('New AGV foundings')
axes[-1].legend(loc='upper left')
fig.suptitle(
    'Density-dependent Poisson founding-rate model across views',
    y=1.02, fontsize=13,
)
plt.tight_layout()
plt.savefig(FIGURES / 'density_dependence_poisson.pdf',
            bbox_inches='tight')
plt.show()


## 8. Limitations

- **Small sample.** The dataset currently spans about a decade; Poisson GLM coefficients are underpowered.
- **Heterogeneous `establishment_basis`.** `founded_date` mixes `announced` / `first_meeting` / `formally_constituted` / `first_output` across venues (§3.2). A stricter analysis would stratify.
- **Census incompleteness.** The v0.3 seed of 106 venues is not a complete enumeration — gaps in, e.g., regional LAC and AU bodies tilt the population sketch.
- **Lifecycle approximation.** Treating `absorbed` / `succeeded` as alive is defensible for institutional continuity but inflates density relative to a strict ecological reading.

These caveats belong in any paper that draws on the fits above; the SSRN companion paper is the right place to work through them carefully.